# 📊 Trabalho Final: Redes Neurais Profundas — **Etapa 3**
**Universidade Federal de Goiás (UFG) — Instituto de Informática (INF)**

**Projeto:** Análise Arquitetural do Congelamento de Camadas na Mitigação de *Domain Shift* e *Language Shift* em Transformers Multilíngues

**Equipe:** Geovana Teixeira Camargo · Sebastião Corrêa Fraga Neto · Pedro Reis Pimenta · Henrique Matheus Mendonça de Miranda

---
> **Status:** **Etapa 3 — Pipeline de Treino Multi-Seed.** O notebook **clona o repositório** (código-fonte `src/`) e lê os 5 CSVs da Etapa 1 a partir do **Google Drive**. Treina os 12 modelos (4 configs × 3 seeds), avalia inline em T1–T4 e gera o `results.csv`. Segue o `docs/PLAN-Etapa3.md`. **Conteúdo atual: Fase 3.0.**

---
## 🟧 Etapa 3 — Treino dos 12 modelos (4 configs × 3 seeds)

| Fase | Descrição | Status |
|------|-----------|--------|
| **3.0** | Setup: clone do repo + Drive + carga dos dados + split `val'` + tokenização | 🟢 **esta entrega** |
| 3.1 | Componentes do `Trainer` (`compute_metrics`, `TrainingArguments`) → `src/train.py` | ⏳ |
| 3.2 | `treinar_run(config, seed)`: treino + avaliação inline em T1–T4 | ⏳ |
| 3.3 | Loop dos 12 treinos (resumível) → `results.csv` (48 linhas) | ⏳ |
| 3.4 | Sanity das curvas de loss | ⏳ |

**Decisões travadas:** HF `Trainer` (D3) · avaliação inline + descarta checkpoint (D4) · `val'` separado do `S1_train`, `S1_val` intocado como T1 (D5) · notebook novo (D6) · `src/` via **git clone** do GitHub.

### 3.0 — Setup, carga dos dados e split de validação

- **INPUT:** repositório no GitHub (`src/`) + 5 CSVs no Drive (`MyDrive/TrabalhoRNP/data_processed/`).
- **AÇÕES:** instalar libs · clonar o repo · seeds + GPU · montar Drive · carregar os 5 CSVs · **split D5** (`S1_train` → `train'` 90% / `val'` 10%, estratificado, seed fixa) · tokenizar (`max_len=128`).
- **OUTPUT:** `ds_train`, `ds_val` tokenizados + `ds_testes` (T1–T4 tokenizados, reusados nas 48 avaliações).
- **VERIFY:** tamanhos batem (`train'` ≈ 9,6k; `val'` ≈ 1,1k; cada célula ≈ 2,68k) e classes balanceadas.

In [ ]:
# 3.0 (1/7) — Bibliotecas
!pip install -q transformers datasets accelerate evaluate scikit-learn
print("Bibliotecas instaladas.")

In [ ]:
# 3.0 (2/7) — Clona o repositório (código-fonte src/, tests/) do GitHub
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Ricktheus/Trabalho-de-Redes-Neurais-Profundas.git"
DIR_REPO = Path("/content/Trabalho-de-Redes-Neurais-Profundas")

if DIR_REPO.exists():
    subprocess.run(["git", "-C", str(DIR_REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(DIR_REPO)], check=True)

if str(DIR_REPO) not in sys.path:
    sys.path.insert(0, str(DIR_REPO))

for m in ("src", "src.model"):
    sys.modules.pop(m, None)
from src import model as M  # carregar_modelo, freeze_layers, CONFIGS, fixar_seed
print("Repo em:", DIR_REPO)
print("src/model.py OK — configs:", list(M.CONFIGS))

In [ ]:
# 3.0 (3/7) — Hardware, seeds e GPU
import torch
import numpy as np
import pandas as pd
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Hardware:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ Sem GPU — Ambiente de execução > Alterar o tipo de ambiente > GPU.")

SEEDS = [42, 123, 2024]   # seeds dos 12 treinos (Etapa 3)
SEED_SPLIT = 42           # seed FIXA do split train'/val' (mesma para todos os runs)
M.fixar_seed(SEED_SPLIT)
print("Seeds dos treinos:", SEEDS, "| seed do split:", SEED_SPLIT)

In [ ]:
# 3.0 (4/7) — Monta o Drive: dados de entrada + pasta de resultados
from google.colab import drive
drive.mount('/content/drive')

DIR_PROJ = Path('/content/drive/MyDrive/TrabalhoRNP')
DIR_DATA = DIR_PROJ / 'data_processed'
DIR_RES  = DIR_PROJ / 'resultados'
(DIR_RES / 'runs').mkdir(parents=True, exist_ok=True)

assert DIR_DATA.exists(), f"Sem {DIR_DATA} — confira a persistência da Etapa 1/2 no Drive."
print("Dados  :", DIR_DATA)
print("Saídas :", DIR_RES, "(runs/ + results.csv chegam na 3.2/3.3)")

In [ ]:
# 3.0 (5/7) — Carrega os 5 CSVs
ARQ = {
    "S1_train": "S1_train_en_eletronicos.csv",
    "S1_val":   "S1_val_en_eletronicos.csv",
    "S2":       "S2_en_beleza.csv",
    "S3":       "S3_pt_eletronicos.csv",
    "S4":       "S4_pt_beleza.csv",
}
dados = {k: pd.read_csv(DIR_DATA / v) for k, v in ARQ.items()}
for k, df in dados.items():
    print(f"{k:>9}: {len(df):>7,} linhas | colunas = {list(df.columns)}")

In [ ]:
# 3.0 (6/7) — Split D5 (val' do S1_train; S1_val intocado = T1) + tokenização
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

# D5: separa 10% do S1_train como val' (só early stopping). S1_val NÃO é tocado -> vira T1.
train_df, val_df = train_test_split(
    dados["S1_train"], test_size=0.10,
    stratify=dados["S1_train"]["label"], random_state=SEED_SPLIT)
print(f"train': {len(train_df):,} (de {len(dados['S1_train']):,}) | val': {len(val_df):,}")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base", use_fast=True)
MAX_LEN = 128

def tokenizar_df(df):
    ds = Dataset.from_pandas(
        df[["texto", "label"]].rename(columns={"label": "labels"}),
        preserve_index=False)
    return ds.map(lambda b: tokenizer(b["texto"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["texto"])

ds_train = tokenizar_df(train_df)
ds_val   = tokenizar_df(val_df)
# Células de teste T1–T4 tokenizadas UMA vez (reusadas nas 48 avaliações da 3.2/3.3).
ds_testes = {"T1": tokenizar_df(dados["S1_val"]), "T2": tokenizar_df(dados["S2"]),
             "T3": tokenizar_df(dados["S3"]),     "T4": tokenizar_df(dados["S4"])}
print("Tokenização OK | colunas:", ds_train.column_names)

In [ ]:
# 3.0 (7/7) — VERIFY: tamanhos, colunas e balanceamento
assert len(train_df) + len(val_df) == len(dados["S1_train"]), "split não soma o total de S1_train"
need = {"input_ids", "attention_mask", "labels"}

def bal(labels):
    a = np.array(labels); n0 = int((a == 0).sum()); n1 = int((a == 1).sum())
    return n0, n1, ("✅" if n0 == n1 else f"⚠️ {n0}/{n1}")

print(f"{'conjunto':>9} | {'N':>7} | neg/pos")
print("-" * 36)
for nome, ds in [("train'", ds_train), ("val'", ds_val), *ds_testes.items()]:
    assert need.issubset(ds.column_names), f"{nome}: faltam colunas {need - set(ds.column_names)}"
    n0, n1, flag = bal(ds["labels"])
    print(f"{nome:>9} | {len(ds):>7,} | {flag}")

assert len(ds_train) > 8000 and len(ds_val) > 800, "split com tamanho inesperado"
assert len({len(ds) for ds in ds_testes.values()}) == 1, "células de teste com tamanhos diferentes"
print("\n✅ Fase 3.0 OK — dados tokenizados; val' separado e S1_val preservado como T1.")
print("➡️  Próximo: Fase 3.1 — compute_metrics + TrainingArguments em src/train.py.")

In [ ]:
%%writefile /content/Trabalho-de-Redes-Neurais-Profundas/src/train.py
import numpy as np
import evaluate
from transformers import TrainingArguments, EarlyStoppingCallback

# Carrega as métricas oficiais
f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # F1-macro e Accuracy
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]
    
    # F1 por classe (0 = Negativo, 1 = Positivo) para ter precisão granular
    f1_classes = f1_metric.compute(predictions=predictions, references=labels, average=None)["f1"]
    
    return {
        "f1_macro": f1_macro,
        "accuracy": acc,
        "f1_negativo": f1_classes[0],
        "f1_positivo": f1_classes[1]
    }

def criar_training_args(out_dir, seed):
    return TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.10,
        fp16=True,                     # Mix precision (ideal para T4)
        eval_strategy="epoch",         # Avalia validação a cada época concluída
        save_strategy="epoch",         # Salva checkpoints a cada época
        load_best_model_at_end=True,   # Puxa o melhor modelo quando o early stopping ativar
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=seed,
        logging_strategy="epoch",      # Log de loss
        report_to="none"               # Desabilita integrações com wandb para não pedir login
    )

def get_early_stopping():
    return EarlyStoppingCallback(early_stopping_patience=1)


In [ ]:
import json
import shutil
import gc
import torch
import pandas as pd
from pathlib import Path
from transformers import Trainer, DataCollatorWithPadding

import sys
if "src.train" in sys.modules:
    del sys.modules["src.train"]
from src import train as T
from src import model as M

RESULTS_CSV = DIR_RES / "results.csv"

def treinar_run(config, seed):
    print(f"\n{'='*60}\n🚀 Iniciando run: Config = {config} | Seed = {seed}\n{'='*60}")
    
    # 1. Carregar modelo e tokenizer (seed fixada)
    model, tok = M.carregar_modelo(seed)
    params = M.freeze_layers(model, config)
    print(f"[{config}-{seed}] Modelo carregado. Parâmetros treináveis: {params['treinavel']:,} / {params['total']:,}")
    
    # 2. Configurar o Trainer e o DataCollator (Dynamic Padding)
    out_dir = f"/content/tmp_trainer_{config}_{seed}"
    args = T.criar_training_args(out_dir, seed)
    data_collator = DataCollatorWithPadding(tokenizer=tok)
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        compute_metrics=T.compute_metrics,
        callbacks=[T.get_early_stopping()],
        data_collator=data_collator
    )
    
    # 3. Dispara o Treino
    print(f"[{config}-{seed}] Treinamento iniciado...")
    trainer.train()
    
    # 4. Salvar histórico de treino (curvas de loss e épocas) na pasta `runs/`
    best_eval_loss = trainer.state.best_metric
    hist = {
        "config": config,
        "seed": seed,
        "best_eval_loss": best_eval_loss,
        "log_history": trainer.state.log_history
    }
    json_path = DIR_RES / f"runs/run_{config}_{seed}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(hist, f, indent=2)
    print(f"[{config}-{seed}] Curvas de perda salvas em {json_path.name}")
    
    # 5. Avaliação inline em T1-T4 (D4: não guardaremos o modelo, só os resultados)
    print(f"[{config}-{seed}] Avaliando o modelo treinado nos testes T1 a T4...")
    linhas_resultado = []
    
    for teste_nome, ds_teste in ds_testes.items():
        metrics = trainer.evaluate(eval_dataset=ds_teste, metric_key_prefix="eval")
        
        linha = {
            "config": config,
            "seed": seed,
            "teste": teste_nome,
            "f1_macro": metrics.get("eval_f1_macro"),
            "accuracy": metrics.get("eval_accuracy"),
            "f1_negativo": metrics.get("eval_f1_negativo"),
            "f1_positivo": metrics.get("eval_f1_positivo")
        }
        linhas_resultado.append(linha)
    
    # Append seguro das 4 linhas no CSV de resultados (cria ou atualiza arquivo)
    df_novo = pd.DataFrame(linhas_resultado)
    if RESULTS_CSV.exists():
        df_novo.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_novo.to_csv(RESULTS_CSV, mode="w", header=True, index=False)
    print(f"[{config}-{seed}] ✅ 4 resultados adicionados em results.csv")
    
    # 6. Apagar checkpoint local para não estourar o disco (D4)
    if Path(out_dir).exists():
        shutil.rmtree(out_dir)
        print(f"[{config}-{seed}] 🗑️ Checkpoints temporários descartados.\n")
        
    # Limpa GPU para garantir que o Colab não sofra OOM nos próximos ciclos
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# FASE 3.3 - LOOP DOS 12 TREINOS (RESUMÍVEL)
# ==========================================
def executar_pipeline():
    runs_feitos = set()
    if RESULTS_CSV.exists():
        df_res = pd.read_csv(RESULTS_CSV)
        for _, row in df_res.iterrows():
            runs_feitos.add((row["config"], row["seed"]))
    
    configs = ["C1", "C2", "C3", "C4"]
    for config in configs:
        for seed in SEEDS:
            if (config, seed) in runs_feitos:
                print(f"⏩ Pulando run já processado: {config} / seed={seed}")
                continue
            
            treinar_run(config, seed)
            
    print("\n🎉 Pipeline da Etapa 3 concluído com sucesso!")

# Chamada principal: inicia os treinamentos
executar_pipeline()
